# LLM-Proto: Build Your Own LLM From Scratch

This notebook validates the full pipeline end-to-end on a **tiny model** (~30M params).  
Designed to run in Google Colab (T4 GPU) or locally.

**Pipeline:**
1. Install dependencies
2. Train BPE tokenizer on sample data
3. Tokenize & save data as binary shards
4. Build model & inspect architecture
5. Run a mini training loop
6. Generate text samples
7. Visualize model internals

---

In [ ]:
# ── Cell 1: Install Dependencies ──
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in [
    "torch>=2.1.0", "tokenizers", "datasets", "wandb", "safetensors",
    "matplotlib", "seaborn", "scikit-learn", "numpy", "pyyaml", "tqdm",
    "google-api-python-client>=2.100.0", "google-auth>=2.23.0",
]:
    install(pkg)

print("All dependencies installed!")

All dependencies installed!


In [ ]:
# ── Cell 2: Imports, Setup & Settings ──
import os, sys, torch, math, time
import numpy as np
from contextlib import nullcontext

# Add project root to path so we can import src/
if os.path.basename(os.getcwd()) != "LLM-Proto":
    if "google.colab" in sys.modules:
        !git clone https://github.com/VlSePr/LLM-Proto
        os.chdir("LLM-Proto")

from src.config import ModelConfig, TrainConfig, MODEL_CONFIGS, get_model_config
from src.model import TransformerLM
from src.tokenizer import LLMTokenizer
from src.utils import (
    detect_environment, get_device, get_dtype, set_seed,
    get_lr, save_checkpoint, load_checkpoint,
)

# ─────────────────────────────────────────
# Settings  (edit these to configure the run)
# ─────────────────────────────────────────
MODEL_SIZE      = "tiny"            # "tiny", "small", "medium", "base", "large"
TOKENIZER_PATH  = "tokenizer_data"
DATA_DIR        = "data"
DATA_CONFIG     = "configs/data.yaml"
CKPT_DIR        = "checkpoints"
N_STEPS         = 100               # mini-training steps (for notebook validation)

# Google Drive checkpoint backup / resume
ENABLE_GDRIVE       = False
GDRIVE_FOLDER_ID    = ""            # folder ID from Drive URL
GDRIVE_CREDENTIALS  = ""            # path to service-account .json (empty = default creds)
RESUME_FROM         = ""            # "" = fresh start, "latest", "best", or "step_N"

# ─────────────────────────────────────────

env = detect_environment()
device = get_device()
dtype = get_dtype()
print(f"Environment: {env}")
print(f"Device: {device}")
print(f"Dtype: {dtype}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Environment: colab
Device: cuda
Dtype: torch.bfloat16
PyTorch: 2.10.0+cu128
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


In [27]:
# ── Cell 3: Inspect All Model Configs ──
print("Available model configurations:\n")
for name, cfg in MODEL_CONFIGS.items():
    est = cfg.param_count_estimate()
    print(f"  {name:8s} | dim={cfg.dim:5d} | layers={cfg.n_layers:2d} | "
          f"heads={cfg.n_heads:2d}/{cfg.n_kv_heads} | ffn={cfg.ffn_dim:5d} | "
          f"seq={cfg.max_seq_len:4d} | ~{est/1e6:.0f}M params")

Available model configurations:

  tiny     | dim=  512 | layers= 6 | heads= 8/4 | ffn= 1536 | seq=2048 | ~34M params
  small    | dim=  768 | layers=12 | heads=12/4 | ffn= 2048 | seq=2048 | ~93M params
  medium   | dim= 1024 | layers=24 | heads=16/4 | ffn= 2816 | seq=2048 | ~278M params
  base     | dim= 1280 | layers=24 | heads=20/4 | ffn= 3584 | seq=2048 | ~426M params
  large    | dim= 2048 | layers=32 | heads=32/8 | ffn= 5632 | seq=4096 | ~1374M params


In [ ]:
# ── Cell 4: Build Model & Verify ──
set_seed(42)
model_config = get_model_config(MODEL_SIZE)
model = TransformerLM(model_config).to(device)
print(model.summary())

# Verify forward pass with random data
dummy_ids = torch.randint(0, model_config.vocab_size, (2, 128), device=device)
dummy_targets = torch.randint(0, model_config.vocab_size, (2, 128), device=device)

with torch.no_grad():
    out = model(dummy_ids, targets=dummy_targets)

print(f"\nForward pass OK!")
print(f"  Logits shape: {out['logits'].shape}")
print(f"  Loss: {out['loss'].item():.4f}")
print(f"  Expected initial loss (ln(vocab)): {np.log(model_config.vocab_size):.4f}")

TransformerLM Summary
  Layers: 6
  Hidden dim: 512
  Attention heads: 8 (KV heads: 4)
  Head dim: 64
  FFN dim: 1536
  Vocab size: 32000
  Max seq len: 2048
  Tie embeddings: True
  Total params: 35,265,024 (35.3M)
  Trainable params: 35,265,024 (35.3M)

Forward pass OK!
  Logits shape: torch.Size([2, 128, 32000])
  Loss: 10.4946
  Expected initial loss (ln(vocab)): 10.3735


In [29]:
# ── Cell 5: Test Generation (random weights — gibberish expected) ──
prompt_ids = torch.tensor([[1, 100, 200, 300]], dtype=torch.long, device=device)  # Dummy prompt
generated = model.generate(prompt_ids, max_new_tokens=20, temperature=0.8, top_k=50)
print(f"Generated token IDs: {generated[0].tolist()}")
print(f"(Gibberish expected — model is untrained)")

Generated token IDs: [1, 100, 200, 300, 13739, 21113, 23738, 20605, 23456, 21542, 20605, 8450, 15567, 14896, 20180, 2622, 20385, 16124, 31557, 31557, 31557, 17051, 9746, 10581]
(Gibberish expected — model is untrained)


In [ ]:
# ── Cell 6: Load Tokenizer ──
# To train:  python scripts/train_tokenizer.py [--config configs/data.yaml] [--force]

if not os.path.exists(os.path.join(TOKENIZER_PATH, "tokenizer.json")):
    raise FileNotFoundError(
        f"No tokenizer found at {TOKENIZER_PATH}/tokenizer.json.\n"
        "Train it first:  python scripts/train_tokenizer.py"
    )

tok = LLMTokenizer(TOKENIZER_PATH)

test_text = "Hello, world! This is a test of the tokenizer."
ids = tok.encode(test_text, add_bos=True, add_eos=True)
decoded = tok.decode(ids)
print(f"Tokenizer loaded (vocab_size={tok.vocab_size})")
print(f"  Input:   {test_text}")
print(f"  Tokens:  {ids[:20]}{'...' if len(ids) > 20 else ''}")
print(f"  Decoded: {decoded}")

Tokenizer already exists at tokenizer_data, skipping training.
Delete the directory to retrain.

Tokenizer test:
  Input:   Hello, world! This is a test of the tokenizer.
  Tokens:  [0, 44, 13724, 16, 913, 5, 662, 315, 262, 1103, 285, 266, 19447, 6017, 18, 1]
  Decoded: Hello, world! This is a test of the tokenizer.
  Vocab size: 32000


In [ ]:
# ── Cell 7: Tokenize Data → Binary Shards ──
# Uses configs/data.yaml to pull from HuggingFace + local custom data.
# Edit data.yaml to add/remove sources and tune processing params.

from src.data import tokenize_and_save

if os.path.exists(os.path.join(DATA_DIR, "train_0000.bin")):
    print(f"Tokenized data already exists in {DATA_DIR}, skipping.")
    print("Delete the .bin files to re-tokenize.")
else:
    print(f"Tokenizing data (config: {DATA_CONFIG})...")
    tokenize_and_save(
        tokenizer_path=TOKENIZER_PATH,
        output_dir=DATA_DIR,
        config_path=DATA_CONFIG,
    )
    print("Data tokenization complete!")

Tokenized data already exists in data, skipping.
Delete the directory to re-tokenize.


In [32]:
# ── Cell 8: Verify DataLoader ──
from src.data import create_dataloader

train_loader = create_dataloader(DATA_DIR, model_config.max_seq_len, batch_size=4, split="train", num_workers=0)

batch = next(iter(train_loader))
input_ids, targets = batch

print(f"Batch input_ids shape: {input_ids.shape}")
print(f"Batch targets shape:   {targets.shape}")
print(f"Sample tokens: {input_ids[0, :20].tolist()}")
print(f"Decoded: {tok.decode(input_ids[0, :50].tolist())[:200]}")

Created train DataLoader: 2,441 samples, 5,000,000 tokens
Batch input_ids shape: torch.Size([4, 2048])
Batch targets shape:   torch.Size([4, 2048])
Sample tokens: [1110, 7365, 6684, 337, 477, 9913, 292, 266, 14749, 23638, 266, 22818, 1848, 288, 266, 5782, 16, 541, 1007, 448]
Decoded:  state officers entered on their duties and the legislature ratified the congressional changes in the constitution, but during this and the next month the legislature proceeded to declare negroes inel


In [ ]:
# ── Cell 8b: (Optional) Resume from Checkpoint ──
# If RESUME_FROM is set in Cell 2, downloads the checkpoint from
# Google Drive (if missing locally) and loads it into a fresh model.
# The training cell below will pick up from start_step.

start_step = 0  # default: train from scratch

if RESUME_FROM:
    print(f"Resuming from '{RESUME_FROM}'...")

    gdrive_kw = {}
    if ENABLE_GDRIVE and GDRIVE_FOLDER_ID:
        gdrive_kw = dict(gdrive_folder_id=GDRIVE_FOLDER_ID,
                         gdrive_credentials_path=GDRIVE_CREDENTIALS)

    model_config = get_model_config(MODEL_SIZE)
    model = TransformerLM(model_config).to(device)
    ckpt = load_checkpoint(CKPT_DIR, RESUME_FROM, model, device=device, **gdrive_kw)
    start_step = ckpt["step"] + 1
    print(f"Restored step {ckpt['step']} (loss={ckpt['loss']:.4f}). Training will resume from step {start_step}.")
else:
    print("Fresh start (RESUME_FROM is empty). Training from scratch.")

In [ ]:
# ── Cell 9: Mini Training Loop (Quick Validation) ──
# Runs a small training loop (~N_STEPS steps) to verify everything works.
# NOT for production — just to confirm loss decreases.

set_seed(42)

# Fresh model (or the one restored by the resume cell above)
if "start_step" not in dir():
    model_config = get_model_config(MODEL_SIZE)
    model = TransformerLM(model_config).to(device)
    start_step = 0

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=6e-4, weight_decay=0.1,
                               betas=(0.9, 0.95), eps=1e-8)

# DataLoader
train_loader = create_dataloader(DATA_DIR, model_config.max_seq_len, batch_size=4,
                                  split="train", num_workers=0)
train_iter = iter(train_loader)

# Mixed precision
use_amp = dtype in (torch.float16, torch.bfloat16) and device.type == "cuda"
amp_ctx = torch.amp.autocast(device_type="cuda", dtype=dtype) if use_amp else nullcontext()
scaler = torch.amp.GradScaler(enabled=(use_amp and dtype == torch.float16))

# Training
model.train()
losses = []

print(f"Mini training: {N_STEPS} steps (starting from {start_step}), batch=4, seq={model_config.max_seq_len}")
print(f"AMP: {use_amp}, dtype: {dtype}\n")

t0 = time.time()
for step in range(start_step, N_STEPS):
    try:
        input_ids, targets = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        input_ids, targets = next(train_iter)

    input_ids = input_ids.to(device)
    targets = targets.to(device)

    lr = get_lr(step, warmup_steps=10, max_steps=N_STEPS, peak_lr=6e-4, min_lr=6e-5)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    optimizer.zero_grad(set_to_none=True)
    with amp_ctx:
        out = model(input_ids, targets=targets)
        loss = out["loss"]

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

    losses.append(loss.item())
    if step % 10 == 0:
        print(f"  step {step:3d} | loss {loss.item():.4f} | ppl {math.exp(min(loss.item(), 20)):.1f} | lr {lr:.2e}")

elapsed = time.time() - t0
print(f"\nDone! {len(losses)} steps in {elapsed:.1f}s")
print(f"Loss: {losses[0]:.4f} → {losses[-1]:.4f} (Δ = {losses[0] - losses[-1]:.4f})")
print(f"✓ Loss is decreasing — pipeline works!" if losses[-1] < losses[0] else "✗ Loss did not decrease — check for bugs")

Created train DataLoader: 2,441 samples, 5,000,000 tokens
Mini training: 100 steps, batch=4, seq=2048
AMP: True, dtype: torch.bfloat16

  step   0 | loss 10.4954 | ppl 36148.0 | lr 6.00e-05
  step  10 | loss 8.6266 | ppl 5577.8 | lr 6.00e-04
  step  20 | loss 7.6929 | ppl 2192.8 | lr 5.84e-04
  step  30 | loss 7.5245 | ppl 1852.9 | lr 5.37e-04
  step  40 | loss 7.4476 | ppl 1715.7 | lr 4.65e-04
  step  50 | loss 7.3451 | ppl 1548.6 | lr 3.77e-04
  step  60 | loss 7.3236 | ppl 1515.6 | lr 2.83e-04
  step  70 | loss 7.1331 | ppl 1252.7 | lr 1.95e-04
  step  80 | loss 7.1465 | ppl 1269.6 | lr 1.23e-04
  step  90 | loss 7.2976 | ppl 1476.7 | lr 7.63e-05

Done! 100 steps in 3.6s
Loss: 10.4954 → 7.0200 (Δ = 3.4753)
✓ Loss is decreasing — pipeline works!


In [34]:
# ── Cell 10: Plot Training Loss ──
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(losses, linewidth=1.5)
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("Mini Training Loss Curve")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [35]:
# ── Cell 11: Generate Text After Mini Training ──
model.eval()

prompts = [
    "The meaning of life is",
    "Once upon a time",
    "In the year 2050",
]

for prompt in prompts:
    ids = tok.encode(prompt, add_bos=True)
    input_tensor = torch.tensor([ids], dtype=torch.long, device=device)
    output = model.generate(input_tensor, max_new_tokens=50, temperature=0.8, top_k=50)
    text = tok.decode(output[0].tolist())
    print(f"Prompt: {prompt}")
    print(f"Output: {text[:200]}")
    print("-" * 60)

Prompt: The meaning of life is
Output: The meaning of life is the a that we to that a can that to in a ", is the same, the not they. It is in an a to a own are.
The not to, you to more.
In the in the one for not to
------------------------------------------------------------
Prompt: Once upon a time
Output: Once upon a time of the study, which of the other of a common that the world.
the:5 of a time the are a time in the other to a other, and be the "It the child to be the first is to be of the most
------------------------------------------------------------
Prompt: In the year 2050
Output: In the year 2050 is that is been of the other is to be one in the first in a new-1.
The common about a
-.
-L, the common they as the most be use of the same can be the first in the a
------------------------------------------------------------


In [36]:
# ── Cell 12: Visualize Model Internals ──
from src.visualize import (
    plot_weight_distributions,
    plot_activation_stats,
    plot_token_loss_heatmap,
)

# Weight distributions
fig = plot_weight_distributions(model)
plt.show()

# Activation stats
sample_ids = next(iter(train_loader))[0][:1].to(device)
fig = plot_activation_stats(model, sample_ids)
plt.show()

# Per-token loss heatmap
fig = plot_token_loss_heatmap(model, sample_ids, tok, max_len=64)
plt.show()

In [ ]:
# ── Cell 13: Save & Load Checkpoint Test ──
train_config = TrainConfig(use_wandb=False)

# Save
save_checkpoint(model, optimizer, step=N_STEPS - 1, loss=losses[-1],
                model_config=model_config, train_config=train_config,
                checkpoint_dir=CKPT_DIR)
print(f"Checkpoint saved to {CKPT_DIR}/")

# Load into a fresh model
model2 = TransformerLM(model_config).to(device)
ckpt = load_checkpoint(CKPT_DIR, "latest", model2, device=device)
print(f"Loaded checkpoint from step {ckpt['step']}")

# Verify weights match
for (n1, p1), (n2, p2) in zip(model.named_parameters(), model2.named_parameters()):
    assert torch.equal(p1, p2), f"Mismatch at {n1}"
print("✓ Checkpoint save/load verified — weights match!")

Checkpoint saved to checkpoints/
Loading checkpoint from checkpoints/latest.pt...


TypeError: RNG state must be a torch.ByteTensor

In [ ]:
# ── Cell 14: (Optional) Upload Checkpoint to Google Drive ──
# Uploads the latest checkpoint and lists what's already on Drive.
# Set ENABLE_GDRIVE = True in Cell 2 and fill in your folder ID.

if ENABLE_GDRIVE:
    from src.gdrive import upload_to_gdrive, list_remote_checkpoints

    # ── 1. Upload current checkpoint to Drive ──
    ckpt_path = os.path.join(CKPT_DIR, "latest.pt")
    if os.path.exists(ckpt_path):
        file_id = upload_to_gdrive(ckpt_path, GDRIVE_FOLDER_ID, GDRIVE_CREDENTIALS)
        print(f"Uploaded {ckpt_path} → Google Drive (id: {file_id})")
    else:
        print(f"No local checkpoint at {ckpt_path}, skipping upload.")

    # ── 2. List available remote checkpoints ──
    remote_files = list_remote_checkpoints(GDRIVE_FOLDER_ID, GDRIVE_CREDENTIALS)
    print(f"\nCheckpoints on Google Drive ({len(remote_files)}):")
    for f in remote_files:
        print(f"  {f['name']:20s}  {f['createdTime']}")
else:
    print("Google Drive upload skipped (ENABLE_GDRIVE = False in Cell 2)")